# Bab 4 — Analisis Regresi dan Prediksi

**Buku:** *Practical Statistics for Data Scientists* (O'Reilly)

## Ringkasan Bab
Analisis regresi digunakan untuk memodelkan hubungan antara prediktor dan variabel respon kontinu. Bab ini membahas:

1. **Regresi Linear Sederhana dan Berganda**.
2. **Evaluasi Model** (RMSE, R-squared, AIC, dan stepwise selection).
3. **Regresi Berbobot (Weighted Regression)**.
4. **Variabel Kategorikal (Dummy Coding)**.
5. **Interpretasi Koefisien, Multikolinearitas, dan Confounders**.
6. **Diagnostik Model** (Residual, Outliers, Leverage, dan Heteroskedastisitas).
7. **Model Non-linear** (Polinomial, Splines, dan GAM).

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.nonparametric.smoothers_lowess import lowess
from statsmodels.gam.api import GLMGam, BSplines
import matplotlib.pyplot as plt
import seaborn as sns

DATA = Path('..') / 'data'
sns.set_theme(style='whitegrid')

## 1. Regresi Linear Sederhana dan Berganda

**Teori.** Regresi linear memodelkan hubungan linier antara variabel prediktor dan variabel target melalui metode kuadrat terkecil (*least squares*).

In [ ]:
lung = pd.read_csv(DATA / 'LungDisease.csv')
ax = lung.plot.scatter(x='Exposure', y='PEFR', figsize=(4, 4))
plt.tight_layout(); plt.show()

In [ ]:
model = smf.ols('PEFR ~ Exposure', data=lung).fit()
print(f'Intercept: {model.params["Intercept"]:.3f}')
print(f'Exposure:  {model.params["Exposure"]:.3f}')

In [ ]:
fitted = model.fittedvalues
residuals = model.resid

ax = lung.plot.scatter(x='Exposure', y='PEFR', figsize=(4, 4))
ax.plot(lung.Exposure, fitted, color='C1')
for x, yactual, yfitted in zip(lung.Exposure, lung.PEFR, fitted):
    ax.plot((x, x), (yactual, yfitted), '--', color='C2', alpha=0.4)
plt.tight_layout(); plt.show()

In [ ]:
house = pd.read_csv(DATA / 'house_sales.csv', sep='\t')
subset = ['AdjSalePrice', 'SqFtTotLiving', 'SqFtLot', 'Bathrooms', 'Bedrooms', 'BldgGrade']
house[subset].head()

In [ ]:
predictors = ['SqFtTotLiving', 'SqFtLot', 'Bathrooms', 'Bedrooms', 'BldgGrade']
outcome = 'AdjSalePrice'

house_lm = sm.OLS(house[outcome], sm.add_constant(house[predictors])).fit()
print(house_lm.params.round(3))

## 2. Evaluasi Model dan Stepwise Selection

**Teori.** Evaluasi model menggunakan RMSE, R-squared, serta kriteria informasi Akaike (AIC) untuk menyeimbangkan kompleksitas model.

In [ ]:
print(house_lm.summary().tables[1])

In [ ]:
predictors_full = ['SqFtTotLiving', 'SqFtLot', 'Bathrooms', 'Bedrooms', 'BldgGrade',
                   'PropertyType', 'NewConstruction']
X = pd.get_dummies(house[predictors_full], drop_first=True, dtype=float)
X['NewConstruction'] = X['NewConstruction'].astype(float)
y = house[outcome]

def aic_of(cols):
    exog = sm.add_constant(X[cols]) if cols else np.ones((len(y), 1))
    return sm.OLS(y, exog).fit().aic

def stepwise(all_cols):
    selected, best = [], aic_of([])
    while True:
        candidates = []
        for c in all_cols:
            if c not in selected:
                candidates.append((aic_of(selected + [c]), selected + [c], f'+ {c}'))
        for c in selected:
            candidates.append((aic_of([s for s in selected if s != c]), [s for s in selected if s != c], f'- {c}'))
        score, cols, step = min(candidates, key=lambda t: t[0])
        if score >= best - 1e-6:
            return selected, best
        selected, best = cols, score
        print(f'{step:<28s} AIC = {best:,.1f}')

best_vars, best_aic = stepwise(list(X.columns))
print('\nSelected variables:', best_vars)

## 3. Regresi Berbobot dan Variabel Kategorikal

**Teori.** Regresi berbobot (*weighted least squares*) memberikan bobot berbeda pada setiap observasi, sementara variabel kategorikal dikodekan menggunakan variabel *dummy*.

In [ ]:
house['Year'] = house['DocumentDate'].str[:4].astype(int)
house['Weight'] = house['Year'] - 2005

house_wt = sm.WLS(house[outcome], sm.add_constant(house[predictors]), weights=house['Weight']).fit()
pd.DataFrame({'unweighted': house_lm.params, 'weighted': house_wt.params}).round(3)

In [ ]:
formula_cat = 'AdjSalePrice ~ SqFtTotLiving + SqFtLot + Bathrooms + Bedrooms + BldgGrade + PropertyType'
house_cat = smf.ols(formula_cat, data=house).fit()
print(house_cat.params.round(3))

## 4. Diagnostik Model dan Hubungan Non-linear

**Teori.** Diagnostik model memeriksa asumsi residual, pencilan, heteroskedastisitas, serta menangani hubungan non-linear melalui polinomial, spline, dan GAM.

In [ ]:
house_98105 = house.loc[house['ZipCode'] == 98105].copy()
lm_98105 = smf.ols('AdjSalePrice ~ SqFtTotLiving + SqFtLot + Bathrooms + Bedrooms + BldgGrade',
                   data=house_98105).fit()

influence = lm_98105.get_influence()
sresiduals = influence.resid_studentized_internal
top_resid = pd.Series(sresiduals, index=house_98105.index).sort_values().head(1)
print('Most negative standardized residual:', round(top_resid.values[0], 2))
print(house_98105.loc[top_resid.index, ['AdjSalePrice', 'SqFtTotLiving', 'Bathrooms', 'Bedrooms', 'BldgGrade']])
print('Fitted value for that record:', round(lm_98105.fittedvalues[top_resid.index[0]]))

In [ ]:
cooks_d = pd.Series(influence.cooks_distance[0], index=house_98105.index)
leverage = pd.Series(influence.hat_matrix_diag, index=house_98105.index)

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(leverage, sresiduals, s=200 * cooks_d / cooks_d.max() + 5, alpha=0.5)
ax.axhline(-3, color='grey', ls='--'); ax.axhline(3, color='grey', ls='--')
ax.set_xlabel('Leverage (hat value)')
ax.set_ylabel('Standardized residual')
ax.set_title("Point size ~ Cook's distance")
plt.tight_layout(); plt.show()

In [ ]:
fitted = lm_98105.fittedvalues
abs_resid = lm_98105.resid.abs()
smooth = lowess(abs_resid, fitted, frac=0.5)

fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(fitted, abs_resid, alpha=0.4)
ax.plot(smooth[:, 0], smooth[:, 1], color='C1')
ax.set_xlabel('Fitted value'); ax.set_ylabel('Absolute residual')
plt.tight_layout(); plt.show()

In [ ]:
base = 'AdjSalePrice ~ SqFtLot + Bathrooms + Bedrooms + BldgGrade'
poly = smf.ols(base + ' + SqFtTotLiving + I(SqFtTotLiving**2)', data=house_98105).fit()
print(f'Linear     R-squared: {lm_98105.rsquared:.4f}  AIC: {lm_98105.aic:,.0f}')
print(f'Polynomial R-squared: {poly.rsquared:.4f}  AIC: {poly.aic:,.0f}')